# Tutorial 9: Capstone - Reproducible Scientific Pipeline

Estimated time: 45-60 minutes

## Prerequisites
Core environment ready; optional backends installed if you want full backend verification.

## Learning aims
- Primary package aim: run a complete reproducible workflow and collect proof artifacts
- Secondary scientific aim: practice evidence-based scientific reporting with provenance and tests

## Success criteria
- you can hand over a concise reproducibility report that another lab member can rerun


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Pre-flight: check prior tutorial artifacts

This capstone builds on earlier tutorials. The cell below checks which artifacts exist so you know what to expect.

In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Locates the repo root, puts src/ on sys.path, and defines helpers that run
# the `mm` CLI in-process (run_mm_cli) and auxiliary tools like pytest/ruff
# via the active interpreter (run_tool). No shell cells, no PYTHONPATH prefix.
import io
import os
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Run the whole notebook from the repo root so CLI artifacts and any
# CWD-relative registry lookups (e.g. eval_surrogate) resolve consistently.
os.chdir(root)

# Jupyter caches imported modules; clear bayesian_metamodeling so re-runs pick
# up the current local source.
for module_name in list(sys.modules):
    if module_name == "bayesian_metamodeling" or module_name.startswith("bayesian_metamodeling."):
        del sys.modules[module_name]

from bayesian_metamodeling.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm_cli(*args: str, check: bool = True) -> int:
    """Run `mm <args>` in-process; cross-platform, no shell."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    out = stdout_buf.getvalue().strip()
    err = stderr_buf.getvalue().strip()
    if out:
        print(out)
    if err:
        print(err)
    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


def run_tool(*args: str, check: bool = True) -> int:
    """Run an auxiliary tool (pytest, ruff) via the active interpreter, cross-platform."""
    cmd = list(args)
    if cmd and cmd[0] in {"pytest", "ruff"}:
        cmd = [sys.executable, "-m", *cmd]
    print("$", " ".join(args))
    with in_project_root():
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Tool failed ({result.returncode}): {' '.join(args)}")
    return result.returncode


In [ ]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

checks = {
    "Tutorial specs": root / "tutorials" / "specs",
    "Run registry": root / "tmp" / "run_registry.json",
    "Surrogate registry": root / "tmp" / "surrogate_registry.json",
    "Metamodel registry": root / "tmp" / "meta_registry.json",
    "Metamodel samples registry": root / "tmp" / "metamodel_samples_registry.json",
}

print("Pre-flight artifact check:")
for label, p in checks.items():
    exists = p.exists()
    status = "FOUND" if exists else "MISSING (some steps may be skipped)"
    print(f"  {label}: {status}")

## Step 1: Validate model specs and DOE planning


In [ ]:
run_mm_cli('validate', 'tutorials/specs/model.toy.grid.json')
run_mm_cli('validate', 'tutorials/specs/model.biomodels.quick.json')
run_mm_cli('plan', 'tutorials/specs/model.toy.sobol.json')


## Step 2: Repository quality gate


In [ ]:
# Quality gate scoped to the metamodeling package (src/ + tests/).
# `.` would also lint/test the tcr_signaling submodule, which is out of scope here.
run_tool('ruff', 'format', '--check', 'src', 'tests')
run_tool('ruff', 'check', 'src', 'tests')
run_tool('pytest', '-q', '-m', 'not slow', 'tests')

## Step 3: Optional backend verification


In [ ]:
run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'pymc_gp_backend_fit_sample_and_logprob', check=False)
run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'sbi_npe_backend_fit_sample_and_logprob', check=False)


## Step 4: Build a mini report card (graphic)


In [ ]:
import matplotlib.pyplot as plt

labels = ['Spec validation', 'DOE planning', 'Fast tests', 'PyMC check', 'SBI check']
status = [1, 1, 1, 0.5, 0.5]  # adjust manually based on your run outcomes

plt.figure(figsize=(8, 3))
plt.bar(labels, status, color=['tab:green' if s == 1 else 'tab:orange' for s in status])
plt.ylim(0, 1.1)
plt.ylabel('completion status')
plt.title('Capstone reproducibility checklist')
plt.grid(axis='y', alpha=0.3)
plt.show()


## Final deliverable template
Include:
- spec files used,
- run IDs and artifact IDs,
- one scientific interpretation,
- any skipped checks and reasons.

If another lab member can reproduce your outputs from this note, you passed.
